# Can we predict `passed_quiz`?
Rows are tutoring sessions; `student_key` repeats across rows and there is no date/order column. So the split unit is the **student**: the test set holds whole students, and cross-validation uses whole-student folds (`GroupKFold`).

**Feature assumptions (columns.csv):** `student_key` is an identifier and is dropped. `enrol_year`, `school_code`, `prior_score` are known before the session. `session_minutes` and `hints_used` are recorded during the session, which precedes the closing quiz, so I treat them as known at prediction time (prediction is made when the quiz starts). If the model must predict before the session starts, these two must be dropped; a pre-session variant is included for CV comparison only.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, accuracy_score, f1_score
df = pd.read_csv('data.csv')
print(df.shape, df.isna().sum().sum(), 'missing;', df.student_key.nunique(), 'students; positive rate', round(df.passed_quiz.mean(), 3))

## 1. Hold out a test set of whole students (before any exploration or modelling)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, te_idx = next(gss.split(df, groups=df.student_key))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
assert not set(train.student_key) & set(test.student_key)
print(len(train), len(test), round(train.passed_quiz.mean(), 3), round(test.passed_quiz.mean(), 3))

## 2. Candidate models (named before any is scored)
1. `baseline`: predict training prevalence (DummyClassifier).
2. `logit_nonid_C*`: standardised logistic regression on `prior_score, session_minutes, hints_used`.
3. `logit_all_C*`: logistic regression on the above plus `enrol_year, school_code` (numeric).
4. `hgb_d*`: HistGradientBoosting on all five features, `school_code` categorical.
5. `logit_noSession_C*`: logistic on `prior_score, enrol_year` only (pre-session sensitivity variant; not eligible for selection).

Hyperparameters: logistic `C` in {0.01, 0.1, 1, 10}; HGB `max_depth` in {2, 3}, learning rate 0.05, 100 iterations. Selection metric: mean out-of-fold ROC AUC. The decision threshold is chosen on out-of-fold predictions to maximise F1.

In [ ]:
num = ['prior_score','session_minutes','hints_used']
feat = ['enrol_year','school_code','prior_score','session_minutes','hints_used']
def logit(cols, C):
    return make_pipeline(ColumnTransformer([('n', StandardScaler(), cols)]), LogisticRegression(C=C, max_iter=1000))
def hgb(depth):
    return HistGradientBoostingClassifier(max_depth=depth, learning_rate=0.05, max_iter=100,
                                          categorical_features=[1], random_state=0)
cands = {'baseline': lambda: DummyClassifier(strategy='prior')}
for C in [0.01, 0.1, 1, 10]:
    cands[f'logit_nonid_C{C}'] = (lambda C=C: logit(num, C))
    cands[f'logit_all_C{C}'] = (lambda C=C: logit(feat, C))
    cands[f'logit_noSession_C{C}'] = (lambda C=C: logit(['prior_score','enrol_year'], C))
for d in [2, 3]:
    cands[f'hgb_d{d}'] = (lambda d=d: hgb(d))
X, y, g = train[feat], train.passed_quiz.values, train.student_key.values

## 3. Grouped cross-validation on training rows only (scalers are inside the per-fold pipelines)

In [ ]:
folds = list(GroupKFold(n_splits=5).split(X, y, g))
oof, rows = {}, []
for name, mk in cands.items():
    p = np.zeros(len(y)); aucs = []
    for a, b in folds:
        m = mk().fit(X.iloc[a], y[a])
        p[b] = m.predict_proba(X.iloc[b])[:, 1]
        aucs.append(roc_auc_score(y[b], p[b]))
    oof[name] = p
    rows.append((name, np.mean(aucs), np.std(aucs), brier_score_loss(y, p)))
cv = pd.DataFrame(rows, columns=['model','cv_auc','auc_sd','brier']).sort_values('cv_auc', ascending=False)
cv

In [ ]:
elig = cv[~cv.model.str.contains('noSession') & (cv.model != 'baseline')]
best = elig.iloc[0].model
ths = np.linspace(0.1, 0.9, 81)
f1s = [f1_score(y, oof[best] > t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('chosen:', best, '| threshold:', round(thr, 3), '| CV F1 at threshold:', round(max(f1s), 3))
print('baseline CV AUC:', round(cv[cv.model == 'baseline'].cv_auc.iloc[0], 3), '| best CV AUC:', round(elig.iloc[0].cv_auc, 3), '+/-', round(elig.iloc[0].auc_sd, 3))

## 4. Decision (fixed before touching the test set)
The model, features, hyperparameters and threshold are those printed above (`best`, `thr`). Nothing changes after seeing test results.

## 5. Refit on all training rows, score the test set once

In [ ]:
final = cands[best]().fit(X, y)
pt = final.predict_proba(test[feat])[:, 1]; yt = test.passed_quiz.values
base = cands['baseline']().fit(X, y).predict_proba(test[feat])[:, 1]
rng = np.random.default_rng(0); ids = test.student_key.unique(); grp = test.groupby('student_key').indices
bs = []
for _ in range(1000):
    s = rng.choice(ids, len(ids)); ix = np.concatenate([grp[i] for i in s])
    bs.append(roc_auc_score(yt[ix], pt[ix]))
print(f'HEADLINE test AUC {roc_auc_score(yt, pt):.3f}  (95% student-bootstrap CI {np.percentile(bs, 2.5):.3f}-{np.percentile(bs, 97.5):.3f})')
print(f'Brier {brier_score_loss(yt, pt):.3f} vs baseline {brier_score_loss(yt, base):.3f}')
print(f'At threshold {thr:.2f}: accuracy {accuracy_score(yt, pt > thr):.3f}, F1 {f1_score(yt, pt > thr):.3f}; always-pass F1 {f1_score(yt, np.ones_like(yt)):.3f}, majority accuracy {max(yt.mean(), 1 - yt.mean()):.3f}')

## Interpretation
Read the headline AUC with its interval; the baseline AUC is 0.5 by construction. Caveats: no timestamps, so this generalises to new students from the same population, not to future sessions; the timing assumption for `session_minutes`/`hints_used` stated at the top; the test set was scored once.